# Financial feature testing with logistic regression

`03_financial_analysis.ipynb` suggested which financial features might help. This notebook tests them.

**Question:** which features improve the score on top of embedding + country?

**Model:** logistic regression with `C=3`, the same settings as in `02_final_ensembles.ipynb`. It is cheap enough to try many feature sets, which the full ensemble is not.

**Method:** 5-fold cross-validation on deduplicated data, with the same folds as notebook 02. Every feature set sees the same folds, so we compare them **fold by fold**. One 90/10 split moves by about +/-0.01, which is larger than the gains we expect, so a single split could not answer this.

**Limits:** this tests logistic regression only. kNN ignores these features, and the MLP can use them in non-linear ways, so a feature that helps here still has to be confirmed in the ensemble.

**Runtime:** about 15 minutes.

In [ ]:
import ast
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

DATA_DIR = "../../data/"   # folder with df_train.pkl and df_financials_train.pkl
N_FOLDS = 5
BLUE = "#2a78d6"
GRAY = "#52514e"

## 1. Load the data

Same steps as in notebook 02: remove duplicates once before any split, then line the financial table up with `df`. Companies without financial data get an empty row.

In [ ]:
df = pd.read_pickle(DATA_DIR + "df_train.pkl")
df = df.drop_duplicates(subset="business_description_embedding").reset_index(drop=True)

X = np.array(df["business_description_embedding"].apply(ast.literal_eval).tolist())
label_encoder = LabelEncoder().fit(df["industry"])
y = label_encoder.transform(df["industry"])

df_fin = pd.read_pickle(DATA_DIR + "df_financials_train.pkl").groupby("id").first()
fin = df_fin.reindex(df["id"].values).reset_index(drop=True)

print("embedding shape:", X.shape, "| industries:", len(label_encoder.classes_))
print("companies without financial data:", fin["country_code"].isna().sum())

## 2. Score

This is `confidence_score` from the course skeleton. We also keep F1 and `1 - Brier` separately, because a feature could improve the confidence without changing the predicted classes.

In [ ]:
def evaluate(y_true, P):
    y_pred = P.argmax(axis=1)
    correct = (y_true == y_pred).astype(float)
    f1 = f1_score(y_true, y_pred, average="weighted")
    calibration = 1 - np.mean((P.max(axis=1) - correct) ** 2)
    return {"score": f1 * calibration, "F1": f1, "1 - Brier": calibration}

## 3. Feature groups

Each group below can be switched on or off.

| Group | Columns | Idea from notebook 03 |
|---|---|---|
| `country` | One-hot country | The only extra that clearly helped so far |
| `zero_revenue` | One flag | Strongest link to industry (Cramer's V 0.32) |
| `turnover` | Scaled value + missing flag | Best effect size (0.253) |
| `ebitda_flag` | One flag: no EBITDA although the company has revenue | V 0.225, without the zero-revenue overlap |
| `net_margin` | Scaled value + missing flag | Weakest ratio (0.126) |
| `ebitda_value` | Scaled value + missing flag | 65% of it is filled in, so probably useless |
| `margin_gap` | Scaled `ebitda_margin - net_profit_margin` + missing flag | New idea: interest, tax and depreciation, so capital-heavy or debt-heavy businesses |
| `per_ratio_zero` | Three flags, one per ratio | What notebook 02 does now |
| `no_row` | One flag | What notebook 02 does now. Same as `unknown` country |

Scaling is signed log, then clipping at the 1st and 99th percentile, then median fill, then z-score. As in notebook 02, every statistic is learned in `fit` from training rows only.

In [ ]:
NUMERIC = {"turnover": "asset_turnover", "net_margin": "net_profit_margin",
           "ebitda_value": "ebitda_margin", "margin_gap": "margin_gap"}


def cleaned(fin, name):
    """Original units, with impossible values set to missing."""
    if name == "margin_gap":
        return cleaned(fin, "ebitda_margin") - cleaned(fin, "net_profit_margin")
    v = fin[name].to_numpy(dtype=float).copy()
    if name == "asset_turnover":
        v[v < 0] = np.nan
    v[np.isinf(v)] = np.nan
    return v


def signed_log(v):
    return np.sign(v) * np.log1p(np.abs(v))


def zero_revenue(fin):
    return np.isinf(fin["net_profit_margin"].to_numpy(dtype=float)) | (fin["asset_turnover"].to_numpy(dtype=float) == 0)


class FeatureGroups:
    def __init__(self, groups):
        self.groups = groups

    def fit(self, fin):
        self.stats = {}
        for group in self.groups:
            if group == "country":
                self.encoder = OneHotEncoder(min_frequency=20, handle_unknown="infrequent_if_exist", sparse_output=False)
                self.encoder.fit(fin["country_code"].fillna("unknown").to_numpy().reshape(-1, 1))
            elif group in NUMERIC:
                s = signed_log(cleaned(fin, NUMERIC[group]))
                low, high = np.nanquantile(s, [0.01, 0.99])
                s = np.clip(s, low, high)
                median = np.nanmedian(s)
                s = np.where(np.isnan(s), median, s)
                self.stats[group] = (low, high, median, s.mean(), s.std())
        return self

    def _columns(self, fin, group):
        if group == "country":
            return self.encoder.transform(fin["country_code"].fillna("unknown").to_numpy().reshape(-1, 1))
        if group in NUMERIC:
            low, high, median, mean, std = self.stats[group]
            s = signed_log(cleaned(fin, NUMERIC[group]))
            missing = np.isnan(s)
            s = np.where(missing, median, np.clip(s, low, high))
            return np.column_stack([(s - mean) / std, missing])
        if group == "zero_revenue":
            return zero_revenue(fin).reshape(-1, 1)
        if group == "ebitda_flag":
            has_row = fin["country_code"].notna().to_numpy()
            no_ebitda = ~np.isfinite(fin["ebitda_margin"].to_numpy(dtype=float))
            return (no_ebitda & has_row & ~zero_revenue(fin)).reshape(-1, 1)
        if group == "per_ratio_zero":
            return np.column_stack([np.isinf(fin["net_profit_margin"].to_numpy(dtype=float)),
                                    np.isinf(fin["ebitda_margin"].to_numpy(dtype=float)),
                                    fin["asset_turnover"].to_numpy(dtype=float) == 0])
        if group == "no_row":
            return fin["country_code"].isna().to_numpy().reshape(-1, 1)
        raise ValueError(group)

    def transform(self, fin):
        return np.hstack([self._columns(fin, group).astype(float) for group in self.groups])

## 4. The experiments

The base is **embedding + country**, because that is the model financial features have to beat.

- First, each group is added **on its own**, to see what it adds by itself.
- Then the combinations suggested by notebook 03, and the full set that notebook 02 uses now.

In [ ]:
BASE = "embedding + country"
BEST_GUESS = "+ zero revenue + turnover"

EXPERIMENTS = {
    "embedding only": [],
    BASE: ["country"],
    "+ zero revenue": ["country", "zero_revenue"],
    "+ turnover": ["country", "turnover"],
    "+ EBITDA flag": ["country", "ebitda_flag"],
    "+ net margin": ["country", "net_margin"],
    "+ EBITDA value": ["country", "ebitda_value"],
    "+ margin gap": ["country", "margin_gap"],
    BEST_GUESS: ["country", "zero_revenue", "turnover"],
    "+ zero revenue + turnover + EBITDA flag": ["country", "zero_revenue", "turnover", "ebitda_flag"],
    "all financials as in notebook 02": ["country", "no_row", "net_margin", "ebitda_value", "turnover", "per_ratio_zero"],
}

## 5. Run the 5-fold cross-validation

The folds use the same seed as notebook 02. The extra features are fitted inside each fold, so nothing leaks from the test rows.

In [ ]:
folds = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=0).split(X, y))
rows = []
oof = {}
done, total, start = 0, len(folds) * len(EXPERIMENTS), time.time()

for fold, (train_idx, test_idx) in enumerate(folds):
    for name, groups in EXPERIMENTS.items():
        F_train, F_test = X[train_idx], X[test_idx]
        if groups:
            extra = FeatureGroups(groups).fit(fin.iloc[train_idx])
            F_train = np.hstack([F_train, extra.transform(fin.iloc[train_idx])])
            F_test = np.hstack([F_test, extra.transform(fin.iloc[test_idx])])

        P = LogisticRegression(C=3, max_iter=3000).fit(F_train, y[train_idx]).predict_proba(F_test)
        rows.append({"fold": fold, "features": name, **evaluate(y[test_idx], P)})
        oof.setdefault(name, np.zeros((len(y), P.shape[1])))[test_idx] = P

        done += 1
        minutes = (time.time() - start) / 60
        print(f"[{done}/{total}] fold {fold + 1}: {name} | {minutes:.1f} min, about {minutes / done * (total - done):.1f} min left", flush=True)

results = pd.DataFrame(rows)

## 6. Results

Every number is the mean over 5 folds.

- **gain:** score minus the base, computed fold by fold and then averaged.
- **gain std:** how much that gain varies between folds.
- **folds improved:** in how many of the 5 folds the feature set beat the base.

**How to read it:** a real gain is positive in at least 4 of 5 folds and larger than about 2 x `gain std / sqrt(5)`. Anything else could be noise.

In [ ]:
pivot = results.pivot(index="fold", columns="features", values="score")
gains = pivot.sub(pivot[BASE], axis=0)

summary = results.groupby("features", sort=False)[["score", "F1", "1 - Brier"]].mean()
summary["gain"] = gains.mean()
summary["gain std"] = gains.std()
summary["folds improved"] = (gains > 0).sum()
summary["clear gain"] = (summary["gain"] > 2 * summary["gain std"] / np.sqrt(N_FOLDS)) & (summary["folds improved"] >= 4)
summary.loc[BASE, "clear gain"] = False
display(summary.sort_values("score", ascending=False).round(4))

In [ ]:
table = summary.drop(index=BASE).sort_values("gain")
position = np.arange(len(table))

fig, ax = plt.subplots(figsize=(10, 5))
ax.axvline(0, color=GRAY, linewidth=1)
ax.hlines(position, table["gain"] - table["gain std"], table["gain"] + table["gain std"], color=BLUE, linewidth=2)
ax.plot(table["gain"], position, "o", color=BLUE, markersize=8, markeredgecolor="white", markeredgewidth=1.5)
ax.set_yticks(position, table.index)
ax.set_xlabel("gain in score against embedding + country (dot = mean, line = +/-1 std across folds)")
ax.set_title("What each feature set adds", loc="left")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

## 7. Which industries change?

The score can hide a shift between industries. This compares the per-industry F1 of the base with `BEST_GUESS`, on the out-of-fold predictions. Notebook 03 suggests Materials, Pharma and Financial Services should gain most.

In [ ]:
def f1_per_industry(P):
    return pd.Series(f1_score(y, P.argmax(axis=1), average=None), index=label_encoder.classes_)


by_industry = pd.DataFrame({"base F1": f1_per_industry(oof[BASE]), "new F1": f1_per_industry(oof[BEST_GUESS])})
by_industry["change"] = by_industry["new F1"] - by_industry["base F1"]
display(by_industry.sort_values("change", ascending=False).round(4))

## 8. Your conclusions

Fill these in after running the notebook.

1. **Single features:** which groups give a clear gain on their own?
2. **Prediction:** did `zero_revenue` and `turnover` do best, as notebook 03 suggested?
3. **Drop list:** do `EBITDA value`, `net margin` and the full notebook 02 set add anything beyond the smaller set?
4. **F1 or confidence:** does a gain come from F1 or from `1 - Brier`?
5. **Next step:** is any gain large enough to test in the full ensemble in `02_final_ensembles.ipynb`?